# 01 — Source Profiling and Validation

## Banking Reporting Platform

This notebook is the first pipeline step after data modelling.

The goal is to treat the CSV files as if they were delivered by an external banking source system and answer:

- Did all expected files arrive?
- Do they have the expected columns?
- Are business keys present and unique?
- Are domain values valid?
- Can dates and amounts be parsed correctly?
- Do foreign-key relationships make sense?
- Do the source records obey the business rules defined in the data model?

This notebook **does not clean or load the data yet**. It only profiles the source and identifies issues.

The next notebook, `02_ingest_raw.ipynb`, will load the source records into PostgreSQL's `raw` schema.


## 1. Imports and Project Paths

The source files are deliberately read as strings first.

That matters because profiling should inspect the values **as received** rather than allowing pandas to silently coerce malformed values before we see them.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    """Find the repository root by locating data/raw from the current path upward."""
    start = start.resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Expected a data/raw directory "
        "in the current directory or one of its parents."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DIR}")


Project root: C:\Users\Admin\Projects\banking-reporting-platform
Raw data directory: C:\Users\Admin\Projects\banking-reporting-platform\data\raw


## 2. Define the Expected Source Contract

This is the minimum contract we expect from the four synthetic source extracts.

The contract comes directly from the approved logical model and source-to-target mapping.


In [2]:
EXPECTED_SCHEMAS = {
    "customers": [
        "customer_id",
        "customer_since_date",
        "customer_status",
    ],
    "accounts": [
        "account_id",
        "account_type",
        "account_status",
        "opened_date",
        "closed_date",
    ],
    "customer_accounts": [
        "customer_id",
        "account_id",
        "holder_role",
    ],
    "transactions": [
        "transaction_id",
        "account_id",
        "transaction_timestamp",
        "transaction_type",
        "channel_code",
        "amount",
        "currency_code",
        "status",
    ],
}

EXPECTED_FILES = {
    name: RAW_DIR / f"{name}.csv"
    for name in EXPECTED_SCHEMAS
}

VALID_DOMAINS = {
    "customer_status": {"ACTIVE", "INACTIVE"},
    "account_type": {"TRANSACTION", "SAVINGS"},
    "account_status": {"ACTIVE", "DORMANT", "CLOSED"},
    "holder_role": {"PRIMARY", "JOINT"},
    "transaction_type": {"PURCHASE", "WITHDRAWAL", "DEPOSIT", "TRANSFER"},
    "channel_code": {"APP", "ATM", "CARD"},
    "transaction_status": {"SUCCESSFUL", "FAILED", "REVERSED"},
    "currency_code": {"ZAR"},
}


## 3. Confirm File Delivery

Before profiling content, verify that every expected file exists.


In [3]:
file_check = pd.DataFrame(
    [
        {
            "dataset": name,
            "file": path.name,
            "exists": path.exists(),
        }
        for name, path in EXPECTED_FILES.items()
    ]
)

display(file_check)

missing_files = file_check.loc[~file_check["exists"], "file"].tolist()

if missing_files:
    raise FileNotFoundError(f"Missing expected source files: {missing_files}")


,dataset,file,exists
0,customers,customers.csv,True
1,accounts,accounts.csv,True
2,customer_accounts,customer_accounts.csv,True
3,transactions,transactions.csv,True


## 4. Load the Source Files

All columns are loaded as strings so malformed dates, invalid numbers, and unexpected codes remain visible.


In [ ]:
dfs = {
    name: pd.read_csv(path, dtype="string", keep_default_na=False)
    for name, path in EXPECTED_FILES.items()
}

for name, df in dfs.items():
    print(f"{name:20s} {len(df):>8,} rows x {len(df.columns)} columns")

customers               1,003 rows x 3 columns
accounts                1,253 rows x 5 columns
customer_accounts       1,355 rows x 3 columns
transactions           50,010 rows x 8 columns


## 5. Structural Profile

First inspect row counts, column counts, and whether each file matches the expected schema.


In [5]:
structure_rows = []

for name, df in dfs.items():
    expected = EXPECTED_SCHEMAS[name]
    actual = df.columns.tolist()

    structure_rows.append(
        {
            "dataset": name,
            "rows": len(df),
            "columns": len(df.columns),
            "missing_columns": sorted(set(expected) - set(actual)),
            "unexpected_columns": sorted(set(actual) - set(expected)),
            "schema_matches": set(expected) == set(actual),
        }
    )

structure_profile = pd.DataFrame(structure_rows)
display(structure_profile)


,dataset,rows,columns,missing_columns,unexpected_columns,schema_matches
0,customers,1003,3,[],[],True
1,accounts,1253,5,[],[],True
2,customer_accounts,1355,3,[],[],True
3,transactions,50010,8,[],[],True


## 6. Null and Blank Profile

Because the CSVs were read as strings, blank text values must be checked explicitly.

A blank is not automatically an error for every field. For example, `closed_date` is allowed to be blank for an open account.


In [6]:
blank_profile = []

for dataset, df in dfs.items():
    for column in df.columns:
        blank_count = df[column].fillna("").str.strip().eq("").sum()

        blank_profile.append(
            {
                "dataset": dataset,
                "column": column,
                "blank_rows": int(blank_count),
                "blank_pct": round((blank_count / len(df)) * 100, 2),
            }
        )

blank_profile = pd.DataFrame(blank_profile)
display(blank_profile)


,dataset,column,blank_rows,blank_pct
0,customers,customer_id,0,0.00
1,customers,customer_since_date,0,0.00
2,customers,customer_status,0,0.00
3,accounts,account_id,0,0.00
4,accounts,account_type,0,0.00
5,accounts,account_status,0,0.00
6,accounts,opened_date,0,0.00
7,accounts,closed_date,1186,94.65
8,customer_accounts,customer_id,0,0.00
9,customer_accounts,account_id,0,0.00


## 7. Business-Key Uniqueness

The approved model defines these business keys:

- Customer → `customer_id`
- Account → `account_id`
- Customer Account → (`customer_id`, `account_id`)
- Transaction → `transaction_id`


In [7]:
key_rules = {
    "customers": ["customer_id"],
    "accounts": ["account_id"],
    "customer_accounts": ["customer_id", "account_id"],
    "transactions": ["transaction_id"],
}

duplicate_summary = []

for dataset, keys in key_rules.items():
    df = dfs[dataset]
    duplicate_mask = df.duplicated(subset=keys, keep=False)

    duplicate_summary.append(
        {
            "dataset": dataset,
            "business_key": ", ".join(keys),
            "duplicate_rows": int(duplicate_mask.sum()),
            "duplicate_key_groups": int(
                df.loc[duplicate_mask, keys].drop_duplicates().shape[0]
            ),
        }
    )

duplicate_summary = pd.DataFrame(duplicate_summary)
display(duplicate_summary)

,dataset,business_key,duplicate_rows,duplicate_key_groups
0,customers,customer_id,2,1
1,accounts,account_id,2,1
2,customer_accounts,"customer_id, account_id",2,1
3,transactions,transaction_id,2,1


## 8. Domain Validation

Now compare categorical values with the allowed values from the data model.


In [8]:
domain_checks = [
    ("customers", "customer_status", VALID_DOMAINS["customer_status"]),
    ("accounts", "account_type", VALID_DOMAINS["account_type"]),
    ("accounts", "account_status", VALID_DOMAINS["account_status"]),
    ("customer_accounts", "holder_role", VALID_DOMAINS["holder_role"]),
    ("transactions", "transaction_type", VALID_DOMAINS["transaction_type"]),
    ("transactions", "channel_code", VALID_DOMAINS["channel_code"]),
    ("transactions", "status", VALID_DOMAINS["transaction_status"]),
    ("transactions", "currency_code", VALID_DOMAINS["currency_code"]),
]

domain_summary = []

for dataset, column, allowed_values in domain_checks:
    values = dfs[dataset][column].str.strip().str.upper()
    invalid_mask = ~values.isin(allowed_values)

    domain_summary.append(
        {
            "dataset": dataset,
            "column": column,
            "invalid_rows": int(invalid_mask.sum()),
            "invalid_values": sorted(values[invalid_mask].unique().tolist()),
        }
    )

domain_summary = pd.DataFrame(domain_summary)
display(domain_summary)


,dataset,column,invalid_rows,invalid_values
0,customers,customer_status,1,[PENDING]
1,accounts,account_type,1,[CHEQUE]
2,accounts,account_status,0,[]
3,customer_accounts,holder_role,1,[OWNER]
4,transactions,transaction_type,1,[FEE]
5,transactions,channel_code,1,[BRANCH]
6,transactions,status,1,[PENDING]
7,transactions,currency_code,1,[RAND]


## 9. Data-Type Validation

Source files contain text, but downstream staging tables require real dates, timestamps, and numeric values.

We therefore test whether source values can be converted safely.


In [9]:
customers = dfs["customers"].copy()
accounts = dfs["accounts"].copy()
transactions = dfs["transactions"].copy()

customers["_customer_since_date_parsed"] = pd.to_datetime(
    customers["customer_since_date"],
    errors="coerce",
)

accounts["_opened_date_parsed"] = pd.to_datetime(
    accounts["opened_date"],
    errors="coerce",
)

accounts["_closed_date_parsed"] = pd.to_datetime(
    accounts["closed_date"].replace("", pd.NA),
    errors="coerce",
)

transactions["_transaction_timestamp_parsed"] = pd.to_datetime(
    transactions["transaction_timestamp"],
    errors="coerce",
)

transactions["_amount_parsed"] = pd.to_numeric(
    transactions["amount"],
    errors="coerce",
)

type_validation = pd.DataFrame(
    [
        {
            "dataset": "customers",
            "field": "customer_since_date",
            "invalid_rows": int(customers["_customer_since_date_parsed"].isna().sum()),
        },
        {
            "dataset": "accounts",
            "field": "opened_date",
            "invalid_rows": int(accounts["_opened_date_parsed"].isna().sum()),
        },
        {
            "dataset": "accounts",
            "field": "closed_date",
            "invalid_rows": int(
                (
                    accounts["closed_date"].str.strip().ne("")
                    & accounts["_closed_date_parsed"].isna()
                ).sum()
            ),
        },
        {
            "dataset": "transactions",
            "field": "transaction_timestamp",
            "invalid_rows": int(
                transactions["_transaction_timestamp_parsed"].isna().sum()
            ),
        },
        {
            "dataset": "transactions",
            "field": "amount",
            "invalid_rows": int(transactions["_amount_parsed"].isna().sum()),
        },
    ]
)

display(type_validation)


,dataset,field,invalid_rows
0,customers,customer_since_date,1
1,accounts,opened_date,0
2,accounts,closed_date,0
3,transactions,transaction_timestamp,1
4,transactions,amount,1


## 10. Referential-Integrity Checks

These checks test whether relationships in the source data agree with the model.

Examples:

- every Customer Account customer should exist in Customers
- every Customer Account account should exist in Accounts
- every Transaction account should exist in Accounts


In [10]:
customer_accounts = dfs["customer_accounts"].copy()

customer_ids = set(dfs["customers"]["customer_id"])
account_ids = set(dfs["accounts"]["account_id"])

unknown_customer_mask = ~customer_accounts["customer_id"].isin(customer_ids)
unknown_ca_account_mask = ~customer_accounts["account_id"].isin(account_ids)
unknown_tx_account_mask = ~transactions["account_id"].isin(account_ids)

referential_summary = pd.DataFrame(
    [
        {
            "relationship": "customer_accounts.customer_id -> customers.customer_id",
            "invalid_rows": int(unknown_customer_mask.sum()),
        },
        {
            "relationship": "customer_accounts.account_id -> accounts.account_id",
            "invalid_rows": int(unknown_ca_account_mask.sum()),
        },
        {
            "relationship": "transactions.account_id -> accounts.account_id",
            "invalid_rows": int(unknown_tx_account_mask.sum()),
        },
    ]
)

display(referential_summary)


,relationship,invalid_rows
0,customer_accounts.customer_id -> customers.cus...,1
1,customer_accounts.account_id -> accounts.accou...,1
2,transactions.account_id -> accounts.account_id,1


## 11. Business-Rule Validation

Now test rules that require more than one field or more than one table.


In [11]:
business_rule_results = []

# Account date consistency.
account_date_mask = (
    accounts["_opened_date_parsed"].notna()
    & accounts["_closed_date_parsed"].notna()
    & (accounts["_closed_date_parsed"] < accounts["_opened_date_parsed"])
)

business_rule_results.append(
    {
        "rule": "Account closed_date must not be earlier than opened_date",
        "failed_rows": int(account_date_mask.sum()),
    }
)

# Closed accounts should have a closed date.
closed_without_date_mask = (
    accounts["account_status"].str.upper().eq("CLOSED")
    & accounts["closed_date"].str.strip().eq("")
)

business_rule_results.append(
    {
        "rule": "CLOSED accounts must have closed_date",
        "failed_rows": int(closed_without_date_mask.sum()),
    }
)

# Transaction amount > 0, only among values that successfully parsed.
amount_not_positive_mask = (
    transactions["_amount_parsed"].notna()
    & (transactions["_amount_parsed"] <= 0)
)

business_rule_results.append(
    {
        "rule": "Transaction amount must be greater than zero",
        "failed_rows": int(amount_not_positive_mask.sum()),
    }
)

# Exactly one PRIMARY holder per valid account.
valid_ca_for_holder_check = customer_accounts[
    customer_accounts["customer_id"].isin(customer_ids)
    & customer_accounts["account_id"].isin(account_ids)
    & customer_accounts["holder_role"].str.upper().isin({"PRIMARY", "JOINT"})
].drop_duplicates(subset=["customer_id", "account_id"])

primary_counts = (
    valid_ca_for_holder_check
    .assign(
        is_primary=lambda x: x["holder_role"].str.upper().eq("PRIMARY").astype(int)
    )
    .groupby("account_id", as_index=False)["is_primary"]
    .sum()
)

valid_account_ids_df = pd.DataFrame({"account_id": list(account_ids)})
primary_check = valid_account_ids_df.merge(
    primary_counts,
    on="account_id",
    how="left",
).fillna({"is_primary": 0})

bad_primary_accounts = primary_check["is_primary"].ne(1)

business_rule_results.append(
    {
        "rule": "Every account must have exactly one PRIMARY holder",
        "failed_rows": int(bad_primary_accounts.sum()),
    }
)

# Every account should have at least one valid holder.
holder_counts = (
    valid_ca_for_holder_check.groupby("account_id")
    .size()
    .reindex(list(account_ids), fill_value=0)
)

business_rule_results.append(
    {
        "rule": "Every account must have at least one valid holder",
        "failed_rows": int((holder_counts == 0).sum()),
    }
)

# Transactions must fall within account lifecycle.
account_dates = accounts[
    [
        "account_id",
        "_opened_date_parsed",
        "_closed_date_parsed",
    ]
].drop_duplicates("account_id", keep="first")

tx_lifecycle = transactions.merge(
    account_dates,
    on="account_id",
    how="left",
)

before_open_mask = (
    tx_lifecycle["_transaction_timestamp_parsed"].notna()
    & tx_lifecycle["_opened_date_parsed"].notna()
    & (
        tx_lifecycle["_transaction_timestamp_parsed"].dt.date
        < tx_lifecycle["_opened_date_parsed"].dt.date
    )
)

after_close_mask = (
    tx_lifecycle["_transaction_timestamp_parsed"].notna()
    & tx_lifecycle["_closed_date_parsed"].notna()
    & (
        tx_lifecycle["_transaction_timestamp_parsed"].dt.date
        > tx_lifecycle["_closed_date_parsed"].dt.date
    )
)

business_rule_results.extend(
    [
        {
            "rule": "Transaction must not occur before account opened_date",
            "failed_rows": int(before_open_mask.sum()),
        },
        {
            "rule": "Transaction must not occur after account closed_date",
            "failed_rows": int(after_close_mask.sum()),
        },
    ]
)

business_rule_summary = pd.DataFrame(business_rule_results)
display(business_rule_summary)


,rule,failed_rows
0,Account closed_date must not be earlier than o...,1
1,CLOSED accounts must have closed_date,0
2,Transaction amount must be greater than zero,1
3,Every account must have exactly one PRIMARY ho...,3
4,Every account must have at least one valid holder,2
5,Transaction must not occur before account open...,0
6,Transaction must not occur after account close...,1


## 12. Consolidated Validation Summary

This combines the most important checks into one view.

A non-zero failure count does **not** mean the notebook failed. In this project, a small number of bad source records were intentionally included so the validation and audit workflow has something realistic to detect.


In [12]:
summary_rows = []

for _, row in duplicate_summary.iterrows():
    summary_rows.append(
        {
            "category": "Duplicate business key",
            "dataset_or_rule": row["dataset"],
            "failed_rows": row["duplicate_rows"],
        }
    )

for _, row in domain_summary.iterrows():
    summary_rows.append(
        {
            "category": "Invalid domain value",
            "dataset_or_rule": f'{row["dataset"]}.{row["column"]}',
            "failed_rows": row["invalid_rows"],
        }
    )

for _, row in type_validation.iterrows():
    summary_rows.append(
        {
            "category": "Invalid data type",
            "dataset_or_rule": f'{row["dataset"]}.{row["field"]}',
            "failed_rows": row["invalid_rows"],
        }
    )

for _, row in referential_summary.iterrows():
    summary_rows.append(
        {
            "category": "Referential integrity",
            "dataset_or_rule": row["relationship"],
            "failed_rows": row["invalid_rows"],
        }
    )

for _, row in business_rule_summary.iterrows():
    summary_rows.append(
        {
            "category": "Business rule",
            "dataset_or_rule": row["rule"],
            "failed_rows": row["failed_rows"],
        }
    )

validation_summary = (
    pd.DataFrame(summary_rows)
    .sort_values(
        ["failed_rows", "category", "dataset_or_rule"],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)

display(validation_summary)

print(
    f"Checks with at least one detected issue: "
    f"{(validation_summary['failed_rows'] > 0).sum()} "
    f"of {len(validation_summary)}"
)


,category,dataset_or_rule,failed_rows
0,Business rule,Every account must have exactly one PRIMARY ho...,3
1,Business rule,Every account must have at least one valid holder,2
2,Duplicate business key,accounts,2
3,Duplicate business key,customer_accounts,2
4,Duplicate business key,customers,2
5,Duplicate business key,transactions,2
6,Business rule,Account closed_date must not be earlier than o...,1
7,Business rule,Transaction amount must be greater than zero,1
8,Business rule,Transaction must not occur after account close...,1
9,Invalid data type,customers.customer_since_date,1


Checks with at least one detected issue: 22 of 27


## 13. Inspect Examples of Failed Records

A summary count tells us **that** a rule failed. A data engineer also needs to inspect representative records before deciding how they should be handled downstream.

The examples below deliberately show only a small number of rows.


In [13]:
print("Duplicate transaction IDs")
display(
    transactions.loc[
        transactions.duplicated("transaction_id", keep=False)
    ].head(10)
)

print("\nUnknown transaction account references")
display(
    transactions.loc[unknown_tx_account_mask].head(10)
)

print("\nTransactions with invalid amounts")
display(
    transactions.loc[
        transactions["_amount_parsed"].isna() | amount_not_positive_mask
    ].head(10)
)

print("\nTransactions after account closure")
display(
    tx_lifecycle.loc[after_close_mask].head(10)
)


Duplicate transaction IDs


,transaction_id,account_id,transaction_timestamp,transaction_type,channel_code,amount,currency_code,status,_transaction_timestamp_parsed,_amount_parsed
99,T00000100,A000316,2026-08-12 18:29:46,PURCHASE,APP,46.87,ZAR,SUCCESSFUL,2026-08-12 18:29:46,46.87
50000,T00000100,A000316,2026-08-12 18:29:46,PURCHASE,APP,46.87,ZAR,SUCCESSFUL,2026-08-12 18:29:46,46.87



Unknown transaction account references


,transaction_id,account_id,transaction_timestamp,transaction_type,channel_code,amount,currency_code,status,_transaction_timestamp_parsed,_amount_parsed
50001,T_BAD_ACCOUNT,A999999,2026-08-20 10:15:00,PURCHASE,CARD,450.00,ZAR,SUCCESSFUL,2026-08-20 10:15:00,450.0



Transactions with invalid amounts


,transaction_id,account_id,transaction_timestamp,transaction_type,channel_code,amount,currency_code,status,_transaction_timestamp_parsed,_amount_parsed
50005,T_BAD_AMOUNT_NEG,A000004,2026-08-20 12:00:00,PURCHASE,CARD,-250.00,ZAR,SUCCESSFUL,2026-08-20 12:00:00,-250.0
50009,T_BAD_AMOUNT_TEXT,A000007,2026-08-20 14:00:00,PURCHASE,CARD,not_a_number,ZAR,SUCCESSFUL,2026-08-20 14:00:00,<NA>



Transactions after account closure


,transaction_id,account_id,transaction_timestamp,transaction_type,channel_code,amount,currency_code,status,_transaction_timestamp_parsed,_amount_parsed,_opened_date_parsed,_closed_date_parsed
50008,T_AFTER_CLOSE,A000037,2025-08-16 00:00:00,PURCHASE,CARD,310.00,ZAR,SUCCESSFUL,2025-08-16,310.0,2019-11-05,2025-08-11


## 14. Profiling Conclusion

At this point we have:

- confirmed the four source files arrived
- checked their structures
- profiled blank values
- checked business-key uniqueness
- validated categorical domains
- tested dates, timestamps, and amounts
- checked source relationships
- tested cross-table business rules
- identified representative bad records

No source rows have been changed yet.

### Next step

`02_ingest_raw.ipynb`

The raw-ingestion notebook will:

1. connect to local PostgreSQL
2. load all received source rows into the `raw` schema
3. preserve source values as received
4. add ingestion metadata such as source file and ingestion timestamp

Validation-based rejection happens when we move from `raw` into the validated staging layer.
